# Import Libraries

In [1]:
import time
import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.support.select import Select

# Interacting with Target Website

In [2]:
def wait_for_page_to_load(driver, wait, old_url = None):
    # To check the URL change
    if old_url:
        try:
            wait.until(EC.url_changes(old_url))
        except:
            print("The URL doesn't change within timeout duration.\n")

    # To check the loading state of the webpage
    try:
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
    except:
        print(f"The page \"{driver.title}\" didn't load fully within the given timeout duration.\n")
    else:
        print(f"Successfully moved to : {driver.title}.\n")
        print(f"Current page url: {driver.current_url}.\n")

In [20]:
# options
chrome_options = Options()
chrome_options.add_argument("--disable-http2")
chrome_options.add_argument("--incognito")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--enable-features=NetworkServiceInProcess")
chrome_options.add_argument("--disable-features=NetworkService")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.63 Safari/537.36"
)

driver = webdriver.Chrome(options = chrome_options)
driver.maximize_window()

# explicit wait
wait = WebDriverWait(driver, 15)

# accessing the target webpage
url = "https://www.99acres.com/"
driver.get(url)

# input("To override the bot")
wait_for_page_to_load(driver, wait, old_url = driver.current_url)

# ActionChains
actions = ActionChains(driver)

# Identify and enter text into the search bar
try:
    search_bar = wait.until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="keyword2"]'))
    )
except:
    print("Timeout while locating Search Bar.\n")
else:
    search_bar.send_keys("chennai")
    time.sleep(2)

# Selecting a valid option from a list
try:
    valid_option = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="0"]')))
except:
    print("Timeout while locating valid search option.\n")
else:
    valid_option.click()
    time.sleep(2)

# click on the Search button
try:
    search_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="searchform_search_btn"]')))
except:
    print("Timeout while locating the \"search\" button.\n")
else:
    search_button.click()
    wait_for_page_to_load(driver, wait, old_url = driver.current_url)


# Select the budget button
try:
    max_button = wait.until(EC.element_to_be_clickable((By.XPATH, "/html[1]/body[1]/div[1]/div[1]/div[1]/div[4]/div[2]/div[1]/div[3]/div[1]/div[2]/div[1]/div[3]/div[1]")))
except:
    print("Timeout while locating the \"maximum\" button.\n")
else:
    max_button.click()

# Select the budget_option button
try:
    budget_option = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="lf_budget_max_list"]/li[71]')))
except:
    print("Timeout while locating the \"Budget option\" button.\n")
else:
    budget_option.click()
    time.sleep(2)


# filter results
# 1. Verified
verified = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//div[contains(@data-label,'VERIFIED_NUDGE')]//span[contains(text(),'Verified')]"))
)
verified.click()
time.sleep(1)

# 2. Ready To Move
ready_to_move = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//div[contains(@data-label,'READY_TO_MOVE_NUDGE')]//span[contains(text(),'Ready To Move')]"))
)
ready_to_move.click()
time.sleep(1)

# moving to the right side to unhide remaining filters
while True:
    try:
        filter_right_button = wait.until(
            EC.presence_of_element_located((By.XPATH, "//i[contains(@class,'iconS_Common_24 icon_upArrow cc__rightArrow')]"))
        )
    except:
        print("Timeout because we have uncovered all filters.\n")
        break
    else:
        filter_right_button.click()
        time.sleep(1)

# With Photos
with_photos = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//span[normalize-space()='With Photos']"))
)
with_photos.click()
time.sleep(1)

# With Videos
with_videos = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//span[normalize-space()='With Videos']"))
)
with_videos.click()
time.sleep(1)

# navigate pages and extract data
data = []
page_count = 0
while True:
    page_count += 1
    try:
        time.sleep(2)
        next_page_button = driver.find_element(By.XPATH, "//a[normalize-space()='Next Page >']")
    except:
        print(f"Timeout because we have navigated all the {page_count} pages.\n")
        break
    else:
        try:
            driver.execute_script("window.scrollBy(0, arguments[0].getBoundingClientRect().top - 100);", next_page_button)
            time.sleep(2)
    
            # scraping the data
            rows = driver.find_elements(By.CLASS_NAME, "tupleNew__TupleContent")
            for row in rows:
                # property name
                try:
                    name = row.find_element(By.CLASS_NAME, "tupleNew__headingNrera").text
                except:
                    name = np.nan
                
                # property location
                try:
                    location = row.find_element(By.CLASS_NAME, "tupleNew__propType").text
                except:
                    location = np.nan
                
                # property price
                try:
                    price = row.find_element(By.CLASS_NAME, "tupleNew__priceValWrap").text
                except:
                    price = np.nan
                
                # property area and bhk
                try:
                    elements = row.find_elements(By.CLASS_NAME, "tupleNew__area1Type")
                except:
                    area, bhk = [np.nan, np.nan]
                else:
                    area, bhk = [ele.text for ele in elements]
                    
                properti = {
                    "name": name,
                    "location": location,
                    "price": price,
                    "area": area,
                    "bhk": bhk
                }
                # print(properti)
                data.append(properti)
                # if properti not in data:
                #     print(properti)
                #     data.append(properti)
            
            wait.until(
            EC.element_to_be_clickable((By.XPATH, "//a[normalize-space()='Next Page >']"))
            ).click()
            time.sleep(3)
        except:
            print("Timeout while clicking on \"Next Page\".\n")


# scraping data from the last page
rows = driver.find_elements(By.CLASS_NAME, "tupleNew__TupleContent")
for row in rows:
	# property name
	try:
		name = row.find_element(By.CLASS_NAME, "tupleNew__headingNrera").text
	except:
		name = np.nan

	# property location
	try:
		location = row.find_element(By.CLASS_NAME, "tupleNew__propType").text
	except:
		location = np.nan

	# property price
	try:
		price = row.find_element(By.CLASS_NAME, "tupleNew__priceValWrap").text
	except:
		price = np.nan

	# property area and bhk
	try:
		elements = row.find_elements(By.CLASS_NAME, "tupleNew__area1Type")
	except:
		area, bhk = [np.nan, np.nan]
	else:
		area, bhk = [ele.text for ele in elements]
					
	property = {
		"name": name,
		"location": location,
		"price": price,
		"area": area,
		"bhk": bhk
	}
	data.append(property)

time.sleep(2)
driver.quit()

The URL doesn't change within timeout duration.

Successfully moved to : India Real Estate Property Site - Buy Sell Rent Properties Portal - 99acres.com.

Current page url: https://www.99acres.com/.

The URL doesn't change within timeout duration.

Successfully moved to : Property in Chennai - Real Estate in Chennai.

Current page url: https://www.99acres.com/search/property/buy/chennai?city=32&preference=S&area_unit=1&res_com=R.

Timeout because we have uncovered all filters.

Timeout while clicking on "Next Page".

Timeout while clicking on "Next Page".

Timeout because we have navigated all the 31 pages.



In [21]:
len(data)

168

In [24]:
# # ----- CLEANING THE DATA -----

df_properties = (
	pd
	.DataFrame(data)
	# .drop_duplicates()
# 	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
# 	.assign(
# 		is_starred=lambda df_: df_.name.str.contains("\n").astype(int),
# 		name=lambda df_: (
# 			df_
# 			.name
# 			.str.replace("\n[0-9.]+", "", regex=True)
# 			.str.strip()
# 			.replace("adroit district s", "adroit district's")
# 		),
# 		location=lambda df_: (
# 			df_
# 			.location
# 			.str.replace("chennai", "")
# 			.str.strip()
# 			.str.replace(",$", "", regex=True)
# 			.str.split("in")
# 			.str[-1]
# 			.str.strip()
# 		),
# 		price=lambda df_: (
# 			df_
# 			.price
# 			.str.replace("₹", "")
# 			.apply(lambda val: float(val.replace("lac", "").strip()) if "lac" in val else float(val.replace("cr", "").strip()) * 100)
# 		),
# 		area=lambda df_: (
# 			df_
# 			.area
# 			.str.replace("sqft", "")
# 			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
# 	)
# 	.rename(columns={
# 		"price": "price_lakhs",
# 		"area": "area_sqft"
# 	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

df_properties

,name,location,price,area,bhk
0,Bright stone,"3 BHK Flat in Adyar, Chennai",₹3.73 Cr,"2,000 sqft",3 BHK
1,Neel Kamal Apartment,"2 BHK Flat in Siruseri, Chennai",₹32.74 Lac,885 sqft,2 BHK
2,Neel Kamal Apartment,"2 BHK Flat in Siruseri, Chennai",₹31.86 Lac,885 sqft,2 BHK
3,Bright stone,"3 BHK Flat in Adyar, Chennai",₹3.73 Cr,"2,000 sqft",3 BHK
4,Neel Kamal Apartment,"2 BHK Flat in Siruseri, Chennai",₹32.74 Lac,885 sqft,2 BHK
...,...,...,...,...,...
163,"Kelambakkam, Chennai, Chennai South","3 Bedroom House in Kelambakkam, Chennai",₹79 Lac,"1,420 sqft",3 BHK
164,"Gowriwakkam,Sembakkam, Chennai, Chennai South","3 BHK Builder Floor in Gowriwakkam,Sembakkam, ...",₹80 Lac,"1,310 sqft",3 BHK
165,"Vadaperumbakkam, Chennai, Chennai North","2 Bedroom House in Vadaperumbakkam, Chennai",₹42 Lac,600 sqft,2 BHK
166,"Gerugambakkam, Chennai, Chennai South","2 Bedroom House in Gerugambakkam, Chennai",₹65 Lac,600 sqft,2 BHK


In [25]:
df_properties.info()

<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   name      168 non-null    str  
 1   location  168 non-null    str  
 2   price     168 non-null    str  
 3   area      168 non-null    str  
 4   bhk       168 non-null    str  
dtypes: str(5)
memory usage: 6.7 KB


In [26]:
# # ----- CLEANING THE DATA -----

df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
# 	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
# 	.assign(
# 		is_starred=lambda df_: df_.name.str.contains("\n").astype(int),
# 		name=lambda df_: (
# 			df_
# 			.name
# 			.str.replace("\n[0-9.]+", "", regex=True)
# 			.str.strip()
# 			.replace("adroit district s", "adroit district's")
# 		),
# 		location=lambda df_: (
# 			df_
# 			.location
# 			.str.replace("chennai", "")
# 			.str.strip()
# 			.str.replace(",$", "", regex=True)
# 			.str.split("in")
# 			.str[-1]
# 			.str.strip()
# 		),
# 		price=lambda df_: (
# 			df_
# 			.price
# 			.str.replace("₹", "")
# 			.apply(lambda val: float(val.replace("lac", "").strip()) if "lac" in val else float(val.replace("cr", "").strip()) * 100)
# 		),
# 		area=lambda df_: (
# 			df_
# 			.area
# 			.str.replace("sqft", "")
# 			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
# 	)
# 	.rename(columns={
# 		"price": "price_lakhs",
# 		"area": "area_sqft"
# 	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

df_properties

,name,location,price,area,bhk
0,Bright stone,"3 BHK Flat in Adyar, Chennai",₹3.73 Cr,"2,000 sqft",3 BHK
1,Neel Kamal Apartment,"2 BHK Flat in Siruseri, Chennai",₹32.74 Lac,885 sqft,2 BHK
2,Neel Kamal Apartment,"2 BHK Flat in Siruseri, Chennai",₹31.86 Lac,885 sqft,2 BHK
6,"Marvel Rivervier View County, Manapakkam, Chen...",3 Bedroom House in Marvel Rivervier View Count...,₹2.7 Cr,"1,808 sqft",3 BHK
7,"Kovur, Chennai, Chennai South","3 Bedroom House in Kovur, Chennai",₹1.14 Cr,800 sqft,3 BHK
...,...,...,...,...,...
163,"Kelambakkam, Chennai, Chennai South","3 Bedroom House in Kelambakkam, Chennai",₹79 Lac,"1,420 sqft",3 BHK
164,"Gowriwakkam,Sembakkam, Chennai, Chennai South","3 BHK Builder Floor in Gowriwakkam,Sembakkam, ...",₹80 Lac,"1,310 sqft",3 BHK
165,"Vadaperumbakkam, Chennai, Chennai North","2 Bedroom House in Vadaperumbakkam, Chennai",₹42 Lac,600 sqft,2 BHK
166,"Gerugambakkam, Chennai, Chennai South","2 Bedroom House in Gerugambakkam, Chennai",₹65 Lac,600 sqft,2 BHK


In [27]:
df_properties.info()

<class 'pandas.DataFrame'>
Index: 158 entries, 0 to 167
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   name      158 non-null    str  
 1   location  158 non-null    str  
 2   price     158 non-null    str  
 3   area      158 non-null    str  
 4   bhk       158 non-null    str  
dtypes: str(5)
memory usage: 7.4 KB


In [29]:
df_properties.name.unique()

<StringArray>
[                                           'Bright stone',
                                    'Neel Kamal Apartment',
 'Marvel Rivervier View County, Manapakkam, Chennai South',
                           'Kovur, Chennai, Chennai South',
                                              'Shiva Flat',
                     'Tharapakkam, Chennai, Chennai South',
      'Irandam Kattalai, Thandalam, Chennai, Chennai West',
                                    'Shriram Park 63\n3.8',
                      'Kundrathur, Chennai, Chennai South',
                   'Gerugambakkam, Chennai, Chennai South',
                                         'Hi Living Evita',
                       'Urbanrise Codename New Porur\n3.8',
                                          'Sidharth Crown',
                               'Purva Somerset House\n3.4',
                                       'Hiliving Serenity',
                                'Urbanrise Revolution One',
                   'Valasa

In [30]:
df_properties.loc[lambda df_: df_.name.str.contains("\n")].name.unique()

<StringArray>
[             'Shriram Park 63\n3.8', 'Urbanrise Codename New Porur\n3.8',
         'Purva Somerset House\n3.4',       'Navins Hanging Gardens\n3.8',
     'Navins Starwood Towers 2\n3.7',                   'Fomra Hues\n4.0',
         'Provident Cosmo City\n3.9',               'Brigade Xanadu\n4.0',
            'Jains Inseli Park\n4.0']
Length: 9, dtype: str

In [36]:
# # ----- CLEANING THE DATA -----

df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		)
# 		location=lambda df_: (
# 			df_
# 			.location
# 			.str.replace("chennai", "")
# 			.str.strip()
# 			.str.replace(",$", "", regex=True)
# 			.str.split("in")
# 			.str[-1]
# 			.str.strip()
# 		),
# 		price=lambda df_: (
# 			df_
# 			.price
# 			.str.replace("₹", "")
# 			.apply(lambda val: float(val.replace("lac", "").strip()) if "lac" in val else float(val.replace("cr", "").strip()) * 100)
# 		),
# 		area=lambda df_: (
# 			df_
# 			.area
# 			.str.replace("sqft", "")
# 			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
 )
# 	.rename(columns={
# 		"price": "price_lakhs",
# 		"area": "area_sqft"
# 	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

# df_properties.query("is_starred == 1").name.unique()
df_properties.query("is_starred == 1")

,name,location,price,area,bhk,is_starred
14,Shriram Park 63,"3 BHK Flat in Perungalathur, Chennai",₹1.7 Cr,"1,800 sqft",3 BHK,1
30,Urbanrise Codename New Porur,"3 BHK Flat in Thirumazhisai, Chennai",₹62 Lac,"1,065 sqft",3 BHK,1
32,Purva Somerset House,"4 BHK Flat in Guindy, Chennai South",₹4.47 Cr,"3,501 sqft",4 BHK,1
35,Purva Somerset House,"3 BHK Flat in Guindy, Chennai South",₹4.1 Cr,"1,968 sqft",3 BHK,1
37,Urbanrise Codename New Porur,"3 BHK Flat in Thirumazhisai, Chennai",₹61 Lac,885 sqft,3 BHK,1
39,Purva Somerset House,"3 BHK Flat in Guindy, Chennai South",₹3.5 Cr,"1,968 sqft",3 BHK,1
40,Purva Somerset House,"4 BHK Flat in Guindy, Chennai South",₹4.63 Cr,"3,501 sqft",4 BHK,1
44,Urbanrise Codename New Porur,"1 BHK Flat in Thirumazhisai, Chennai",₹40 Lac,633 sqft,1 BHK,1
45,Shriram Park 63,"3 BHK Flat in Perungalathur, Chennai",₹1.4 Cr,"1,440 sqft",3 BHK,1
53,Urbanrise Codename New Porur,"3 BHK Flat in Thirumazhisai, Chennai",₹70 Lac,"1,065 sqft",3 BHK,1


In [37]:
df_properties.name.unique()

<StringArray>
[                                           'Bright stone',
                                    'Neel Kamal Apartment',
 'Marvel Rivervier View County, Manapakkam, Chennai South',
                           'Kovur, Chennai, Chennai South',
                                              'Shiva Flat',
                     'Tharapakkam, Chennai, Chennai South',
      'Irandam Kattalai, Thandalam, Chennai, Chennai West',
                                         'Shriram Park 63',
                      'Kundrathur, Chennai, Chennai South',
                   'Gerugambakkam, Chennai, Chennai South',
                                         'Hi Living Evita',
                            'Urbanrise Codename New Porur',
                                          'Sidharth Crown',
                                    'Purva Somerset House',
                                       'Hiliving Serenity',
                                'Urbanrise Revolution One',
                   'Valasa

In [38]:
df_properties

,name,location,price,area,bhk,is_starred
0,Bright stone,"3 BHK Flat in Adyar, Chennai",₹3.73 Cr,"2,000 sqft",3 BHK,0
1,Neel Kamal Apartment,"2 BHK Flat in Siruseri, Chennai",₹32.74 Lac,885 sqft,2 BHK,0
2,Neel Kamal Apartment,"2 BHK Flat in Siruseri, Chennai",₹31.86 Lac,885 sqft,2 BHK,0
6,"Marvel Rivervier View County, Manapakkam, Chen...",3 Bedroom House in Marvel Rivervier View Count...,₹2.7 Cr,"1,808 sqft",3 BHK,0
7,"Kovur, Chennai, Chennai South","3 Bedroom House in Kovur, Chennai",₹1.14 Cr,800 sqft,3 BHK,0
...,...,...,...,...,...,...
163,"Kelambakkam, Chennai, Chennai South","3 Bedroom House in Kelambakkam, Chennai",₹79 Lac,"1,420 sqft",3 BHK,0
164,"Gowriwakkam,Sembakkam, Chennai, Chennai South","3 BHK Builder Floor in Gowriwakkam,Sembakkam, ...",₹80 Lac,"1,310 sqft",3 BHK,0
165,"Vadaperumbakkam, Chennai, Chennai North","2 Bedroom House in Vadaperumbakkam, Chennai",₹42 Lac,600 sqft,2 BHK,0
166,"Gerugambakkam, Chennai, Chennai South","2 Bedroom House in Gerugambakkam, Chennai",₹65 Lac,600 sqft,2 BHK,0


In [39]:
df_properties.location

0                           3 BHK Flat in Adyar, Chennai
1                        2 BHK Flat in Siruseri, Chennai
2                        2 BHK Flat in Siruseri, Chennai
6      3 Bedroom House in Marvel Rivervier View Count...
7                      3 Bedroom House in Kovur, Chennai
                             ...                        
163              3 Bedroom House in Kelambakkam, Chennai
164    3 BHK Builder Floor in Gowriwakkam,Sembakkam, ...
165          2 Bedroom House in Vadaperumbakkam, Chennai
166            2 Bedroom House in Gerugambakkam, Chennai
167            3 Bedroom House in Gerugambakkam, Chennai
Name: location, Length: 158, dtype: str

In [41]:
df_properties.location.str.contains("Chennai").value_counts()

location
True     148
False     10
Name: count, dtype: int64

In [42]:
df_properties.location.iloc[0]

'3 BHK Flat in Adyar, Chennai'

In [43]:
# # ----- CLEANING THE DATA -----

df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			# .str.replace(",$", "", regex=True)
			# .str.split("in")
			# .str[-1]
			# .str.strip()
		),
# 		price=lambda df_: (
# 			df_
# 			.price
# 			.str.replace("₹", "")
# 			.apply(lambda val: float(val.replace("lac", "").strip()) if "lac" in val else float(val.replace("cr", "").strip()) * 100)
# 		),
# 		area=lambda df_: (
# 			df_
# 			.area
# 			.str.replace("sqft", "")
# 			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
 )
# 	.rename(columns={
# 		"price": "price_lakhs",
# 		"area": "area_sqft"
# 	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

# df_properties.query("is_starred == 1").name.unique()
df_properties.location.iloc[0]

'3 BHK Flat in Adyar,'

In [51]:
# # ----- CLEANING THE DATA -----

df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			# .str.split("in")
			# .str[-1]
			# .str.strip()
		),
# 		price=lambda df_: (
# 			df_
# 			.price
# 			.str.replace("₹", "")
# 			.apply(lambda val: float(val.replace("lac", "").strip()) if "lac" in val else float(val.replace("cr", "").strip()) * 100)
# 		),
# 		area=lambda df_: (
# 			df_
# 			.area
# 			.str.replace("sqft", "")
# 			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
 )
# 	.rename(columns={
# 		"price": "price_lakhs",
# 		"area": "area_sqft"
# 	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

# df_properties.query("is_starred == 1").name.unique()
df_properties.location.str.split("in").str[-1].str.strip().iloc[0]

'Adyar'

In [53]:
# # ----- CLEANING THE DATA -----

df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
# 		price=lambda df_: (
# 			df_
# 			.price
# 			.str.replace("₹", "")
# 			.apply(lambda val: float(val.replace("lac", "").strip()) if "lac" in val else float(val.replace("cr", "").strip()) * 100)
# 		),
# 		area=lambda df_: (
# 			df_
# 			.area
# 			.str.replace("sqft", "")
# 			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
 )
# 	.rename(columns={
# 		"price": "price_lakhs",
# 		"area": "area_sqft"
# 	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

# df_properties.query("is_starred == 1").name.unique()
df_properties.location.unique()

array(['Adyar', 'Siruseri', 'Marvel Rivervier View County, Manapakkam',
       'Kovur', 'Nanmangalam', 'Tharapakkam',
       'Irandam Kattalai, Thandalam', 'Perungalathur', 'Kundrathur',
       'Gerugambakkam', 'Madhavaram', 'Thirumazhisai', 'dy,  South',
       'Padur', 'Valasaravakkam', 'Porur', 'Medavakkam', 'Mangadu',
       'Iyyappanthangal', 'Kattupakkam', 'Perambur', 'Tambaram',
       'Sendurpuram, Kattupakkam', 'Kathirvedu, Puzhal', 'Egattur, OMR',
       'Maduravoyal', 'Thirumalai Nagar, Sembakkam', 'Koyambedu',
       'Pallikaranai', 'Alwarthirunagar, Valasaravakkam',
       'New perungalathur, Perungalathur', 'Manapakkam', 'Rajakilpakkam',
       'West Tambaram,  South', 'Karpagam Gardens, Adyar', 'Ayappakkam',
       'Perungavoor', 'Kolapakkam', 'Gandhi Nagar,Adyar',
       'Alandur, GST Road', 'Karapakkam, OMR', 'Pudupakkam Village',
       'Keelkattalai', 'apuram, Chromepet', 'Sembakkam', 'Red Hills',
       'Kolathur', 'Mogappair', 'Kelambakkam', 'Gowriwakkam,Sembakkam'

In [55]:
df_properties.price.str[0].unique()

<StringArray>
['₹']
Length: 1, dtype: str

In [60]:
# # ----- CLEANING THE DATA -----

df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
		price=lambda df_: (
			df_
			.price
			.str.replace("₹", "")
# 			.apply(lambda val: float(val.replace("lac", "").strip()) if "lac" in val else float(val.replace("cr", "").strip()) * 100)
		),
# 		area=lambda df_: (
# 			df_
# 			.area
# 			.str.replace("sqft", "")
# 			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
 )
# 	.rename(columns={
# 		"price": "price_lakhs",
# 		"area": "area_sqft"
# 	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

# df_properties.query("is_starred == 1").name.unique()
df_properties.price.str.split(" ").str[-1].unique()

array(['Cr', 'Lac'], dtype=object)

In [66]:
# # ----- CLEANING THE DATA -----

df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
		price=lambda df_: (
			df_
			.price
			.str.replace("₹", "")
			.apply(lambda val: float(val.replace('Lac', "").strip()) if 'Lac' in val else float(val.replace('Cr', "").strip()) * 100)
		),
# 		area=lambda df_: (
# 			df_
# 			.area
# 			.str.replace("sqft", "")
# 			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
 )
	.rename(columns={
		"price": "price_lakhs",
		# "area": "area_sqft"
	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

# df_properties.query("is_starred == 1").name.unique()
# df_properties.price
df_properties.price_lakhs.dtype

dtype('float64')

In [67]:
df_properties.area.unique()

<StringArray>
['2,000 sqft',   '885 sqft', '1,808 sqft',   '800 sqft', '1,100 sqft',
   '838 sqft',   '878 sqft', '1,000 sqft', '1,800 sqft',   '600 sqft',
 ...
 '1,026 sqft',   '525 sqft',   '621 sqft', '1,494 sqft', '1,175 sqft',
 '1,064 sqft',   '960 sqft', '1,420 sqft', '1,310 sqft',   '700 sqft']
Length: 115, dtype: str

In [69]:
df_properties.area.str.contains("sqft").unique()

array([ True])

In [70]:
df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
		price=lambda df_: (
			df_
			.price
			.str.replace("₹", "")
			.apply(lambda val: float(val.replace('Lac', "").strip()) if 'Lac' in val else float(val.replace('Cr', "").strip()) * 100)
		),
		area=lambda df_: (
			df_
			.area
			.str.replace("sqft", "")
			.str.strip()
# 			.str.replace(",", "")
# 			.pipe(lambda ser: pd.to_numeric(ser))
		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
 )
	.rename(columns={
		"price": "price_lakhs",
		# "area": "area_sqft"
	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

In [74]:
df_properties.area.str.contains("[^0-9]", regex = True).value_counts()

area
True     90
False    68
Name: count, dtype: int64

In [77]:
df_properties.loc[lambda df_: df_.area.str.contains("[^0-9]", regex = True)].area.unique()

<StringArray>
['2,000', '1,808', '1,100', '1,000', '1,800', '1,700', '1,200', '1,150',
 '1,728', '1,383', '1,350', '1,065', '1,457', '3,501', '1,300', '1,968',
 '2,400', '1,440', '1,548', '1,110', '1,087', '1,047', '1,495', '1,417',
 '2,596', '1,040', '1,140', '2,330', '6,400', '1,301', '2,308', '1,535',
 '1,250', '1,280', '1,358', '1,050', '1,162', '1,025', '1,234', '1,908',
 '1,090', '1,001', '1,275', '1,400', '1,441', '1,750', '1,182', '1,373',
 '1,375', '1,392', '1,186', '1,525', '1,135', '1,073', '1,291', '1,835',
 '3,096', '1,062', '1,519', '1,539', '1,121', '1,204', '1,203', '1,026',
 '1,494', '1,175', '1,064', '1,420', '1,310']
Length: 69, dtype: str

In [78]:
df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
		price=lambda df_: (
			df_
			.price
			.str.replace("₹", "")
			.apply(lambda val: float(val.replace('Lac', "").strip()) if 'Lac' in val else float(val.replace('Cr', "").strip()) * 100)
		),
		area=lambda df_: (
			df_
			.area
			.str.replace("sqft", "")
			.str.strip()
			.str.replace(",", "")
			.pipe(lambda ser: pd.to_numeric(ser))
		),
# 		bhk=lambda df_: (
# 			df_
# 			.bhk
# 			.str.replace("bhk", "")
# 			.str.strip()
# 			.pipe(lambda ser: pd.to_numeric(ser))
# 		)
 )
	.rename(columns={
		"price": "price_lakhs",
		"area": "area_sqft"
	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

In [82]:
df_properties.area_sqft.dtype

dtype('int64')

In [87]:
df_properties.bhk.str.contains("BHK").unique()

array([ True])

In [94]:
df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
		price=lambda df_: (
			df_
			.price
			.str.replace("₹", "")
			.apply(lambda val: float(val.replace('Lac', "").strip()) if 'Lac' in val else float(val.replace('Cr', "").strip()) * 100)
		),
		area=lambda df_: (
			df_
			.area
			.str.replace("sqft", "")
			.str.strip()
			.str.replace(",", "")
			.pipe(lambda ser: pd.to_numeric(ser))
		),
		bhk=lambda df_: (
			df_
			.bhk
			.str.replace("BHK", "")
			.str.strip()
			# .pipe(lambda ser: pd.to_numeric(ser))
		)
 )
	.rename(columns={
		"price": "price_lakhs",
		"area": "area_sqft"
	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

In [96]:
df_properties.bhk.str.contains("[^0-9]", regex = True).unique()

array([False])

In [97]:
df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
		price=lambda df_: (
			df_
			.price
			.str.replace("₹", "")
			.apply(lambda val: float(val.replace('Lac', "").strip()) if 'Lac' in val else float(val.replace('Cr', "").strip()) * 100)
		),
		area=lambda df_: (
			df_
			.area
			.str.replace("sqft", "")
			.str.strip()
			.str.replace(",", "")
			.pipe(lambda ser: pd.to_numeric(ser))
		),
		bhk=lambda df_: (
			df_
			.bhk
			.str.replace("BHK", "")
			.str.strip()
			.pipe(lambda ser: pd.to_numeric(ser))
		)
 )
	.rename(columns={
		"price": "price_lakhs",
		"area": "area_sqft"
	})
# 	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

In [98]:
df_properties.bhk.dtype

dtype('int64')

In [99]:
df_properties.info()

<class 'pandas.DataFrame'>
Index: 158 entries, 0 to 167
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         158 non-null    str    
 1   location     158 non-null    object 
 2   price_lakhs  158 non-null    float64
 3   area_sqft    158 non-null    int64  
 4   bhk          158 non-null    int64  
 5   is_starred   158 non-null    int64  
dtypes: float64(1), int64(3), object(1), str(1)
memory usage: 8.6+ KB


In [100]:
df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
		price=lambda df_: (
			df_
			.price
			.str.replace("₹", "")
			.apply(lambda val: float(val.replace('Lac', "").strip()) if 'Lac' in val else float(val.replace('Cr', "").strip()) * 100)
		),
		area=lambda df_: (
			df_
			.area
			.str.replace("sqft", "")
			.str.strip()
			.str.replace(",", "")
			.pipe(lambda ser: pd.to_numeric(ser))
		),
		bhk=lambda df_: (
			df_
			.bhk
			.str.replace("BHK", "")
			.str.strip()
			# .pipe(lambda ser: pd.to_numeric(ser))
		)
 )
	.rename(columns={
		"price": "price_lakhs",
		"area": "area_sqft"
	})
	.reset_index(drop=True)
# 	.to_excel("chennai-properties-99acres.xlsx", index=False)
)

df_properties

,name,location,price_lakhs,area_sqft,bhk,is_starred
0,Bright stone,Adyar,373.00,2000,3,0
1,Neel Kamal Apartment,Siruseri,32.74,885,2,0
2,Neel Kamal Apartment,Siruseri,31.86,885,2,0
3,"Marvel Rivervier View County, Manapakkam, Chen...","Marvel Rivervier View County, Manapakkam",270.00,1808,3,0
4,"Kovur, Chennai, Chennai South",Kovur,114.00,800,3,0
...,...,...,...,...,...,...
153,"Kelambakkam, Chennai, Chennai South",Kelambakkam,79.00,1420,3,0
154,"Gowriwakkam,Sembakkam, Chennai, Chennai South","Gowriwakkam,Sembakkam",80.00,1310,3,0
155,"Vadaperumbakkam, Chennai, Chennai North",Vadaperumbakkam,42.00,600,2,0
156,"Gerugambakkam, Chennai, Chennai South",Gerugambakkam,65.00,600,2,0


In [101]:
df_properties = (
	pd
	.DataFrame(data)
	.drop_duplicates()
	.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)
	.assign(
		is_starred = lambda df_: df_.name.str.contains("\n").astype(int),
		name = lambda df_: (
			df_
			.name
			.str.replace("\n[0-9.]+", "", regex=True)
			.str.strip()
			# .replace("adroit district s", "adroit district's")
		),
		location=lambda df_: (
			df_
			.location
			.str.replace("Chennai", "")
			.str.strip()
			.str.replace(",$", "", regex=True)
			.str.split("in")
			.str[-1]
			.str.strip()
		),
		price=lambda df_: (
			df_
			.price
			.str.replace("₹", "")
			.apply(lambda val: float(val.replace('Lac', "").strip()) if 'Lac' in val else float(val.replace('Cr', "").strip()) * 100)
		),
		area=lambda df_: (
			df_
			.area
			.str.replace("sqft", "")
			.str.strip()
			.str.replace(",", "")
			.pipe(lambda ser: pd.to_numeric(ser))
		),
		bhk=lambda df_: (
			df_
			.bhk
			.str.replace("BHK", "")
			.str.strip()
			.pipe(lambda ser: pd.to_numeric(ser))
		)
 )
	.rename(columns={
		"price": "price_lakhs",
		"area": "area_sqft"
	})
	.reset_index(drop=True)
	.to_excel("chennai-properties-99acres.xlsx", index=False)
)